# Train the delivery model on CREMA-D and RAVDESS

Run on Kaggle with a GPU and **Internet on** (needed to `pip install tract`).

Before running:
1. Add these notebook inputs: a RAVDESS audio dataset, a CREMA-D audio dataset, and this repo uploaded as a dataset (it must contain the `btt/` folder).
2. Optional: if one of the inputs contains CREMA-D's `VideoDemographics.csv`, per-group results are computed automatically.
3. Licences: CREMA-D is ODbL, RAVDESS is CC BY-NC-SA 4.0, so the trained model is non-commercial.

The corpus folders are found by walking the input folders, so layout does not matter.

In [ ]:
!pip install -q tract==0.23.8 praat-parselmouth onnx torchview
import glob, os, shutil, sys
btt = next(iter(glob.glob('/kaggle/input/*/btt')), None)
assert btt, 'Add the repo as a dataset containing the btt/ folder.'
shutil.copytree(os.path.dirname(btt), '/kaggle/working/repo', dirs_exist_ok=True)
os.chdir('/kaggle/working/repo'); sys.path.insert(0, '.')
print(os.listdir('.'))

In [ ]:
inputs = sorted(glob.glob('/kaggle/input/*'))
print(inputs)
RAV = next(p for p in inputs if 'ravdess' in p.lower())
CRE = next(p for p in inputs if 'crema' in p.lower())
DEMO = next(iter(glob.glob('/kaggle/input/**/VideoDemographics.csv', recursive=True)), None)
print(RAV, CRE, DEMO)

In [ ]:
import torch
print(torch.__version__, 'cuda:', torch.cuda.is_available())
from btt.train import main
args = ['--ravdess', RAV, '--cremad', CRE, '--out', '/kaggle/working/artifacts', '--epochs', '40', '--mode', 'both']
if DEMO: args += ['--demographics', DEMO]
main(args)

In [ ]:
# PyTorch and tract must agree. This is the guard for tract issue #2751 (GRU).
import json
meta = json.load(open('/kaggle/working/artifacts/meta.json'))
metrics = json.load(open('/kaggle/working/artifacts/metrics.json'))
print(meta['data_source'], metrics['parity_pytorch_vs_tract'])
assert meta['data_source'] == 'real' and metrics['parity_pytorch_vs_tract']['passed']
print(open('/kaggle/working/artifacts/MODEL_CARD.md').read()[:3000])

In [ ]:
!python scripts/arch_diagram.py --out /kaggle/working/arch
!cd /kaggle/working && tar czf artifacts.tgz artifacts arch
# Download artifacts.tgz from the Output tab. The server needs model.onnx, meta.json, metrics.json and MODEL_CARD.md.